# KNN & SVM — Distance-Based Models

**Goal:** Master the two most important instance-based and margin-based classifiers.

**Why it matters:**
- **KNN** — the simplest possible classifier. No training, just memory. A perfect teaching example for the bias-variance tradeoff.
- **SVM** — historically the strongest "shallow" classifier. Foundation for kernel methods and understanding margins.
- Both are **distance-based** → require careful feature scaling.
- Both are used in production when data is small/medium and interpretability is less critical.

**Prerequisite:** [02-tree-randomforest.ipynb](./02-tree-randomforest.ipynb)

**Dataset:** Titanic

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)

print("Setup complete.")

Setup complete.


In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df["HasCabin"] = df["Cabin"].notna().astype(int)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(lambda x: x.fillna(x.median()))
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch",
         "Fare", "Embarked", "HasCabin", "FamilySize", "IsAlone"]]

X = df.drop(columns=["Survived"])
y = df["Survived"]

numeric_features = ["Age", "SibSp", "Parch", "Fare", "HasCabin", "FamilySize", "IsAlone"]
categorical_features = ["Pclass", "Sex", "Embarked"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (712, 10), Test: (179, 10)


In [3]:


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop=None)),
    
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

print("Preprocessor ready (with scaling).")

Preprocessor ready (with scaling).


## PART 1 — K-Nearest Neighbors (KNN)

### The Idea

**Training:** none. Just store the training data.  
**Prediction:** find the `k` nearest training points, take majority vote.

That's it. No parameters to learn, no gradient descent.

**Why it works:** Points close in feature space tend to have the same label.

### The Bias-Variance Tradeoff via `k`

| k | Behavior | Bias | Variance |
|---|----------|------|----------|
| 1 | Follows every point | Low | High (overfit) |
| 5–15 | Smooth decision boundary | Medium | Medium |
| Large | Over-smoothed | High (underfit) | Low |

### Key Hyperparameters

| Parameter | What it does | Options |
|-----------|-------------|---------|
| `n_neighbors` | How many neighbors vote | 1 to √n, tune it |
| `weights` | Uniform or distance-weighted | "uniform", "distance" |
| `metric` | Distance measure | "euclidean", "manhattan", "minkowski" |
| `p` | Minkowski power | 1=manhattan, 2=euclidean |
| `algorithm` | Search method | "auto", "ball_tree", "kd_tree", "brute" |

### Why Scaling is Non-Negotiable for KNN

If feature A ranges [0, 1000] and B ranges [0, 1], the distance is dominated by A.
Scaling forces every feature to contribute fairly.

---